In [1]:
import pandas as pd
import requests
import geopandas as gpd
from shapely.geometry import Point

In [2]:
# Cargar el shapefile de Community Environs (en lugar del de DAs)
gdf_ce = gpd.read_file(r"C:\Users\maria\OneDrive\Escritorio\HealthLocate\shapefiles\ComEnviron.shp")

# Ver qué columnas tiene
print(gdf_ce.columns.tolist())
gdf_ce.head()

['DA_grp', 'comenvnm', 'AREA_KM2', 'NUM_DAs', 'DA16popest', 'DA16pop_m', 'DA16pop_f', 'CHBProp4', 'CHBPropF', 'id', 'region', 'name', 'smk_curr_m', 'smk_currlc', 'smk_curruc', 'smk_curr_1', 'smk_curr_f', 'smk_curr_2', 'smk_curr_3', 'smk_curr_4', 'smk_fmr_ma', 'smk_fmr__1', 'smk_fmr__2', 'smk_fmr__3', 'smk_fmr_fe', 'smk_fmr__4', 'smk_fmr__5', 'smk_fmr__6', 'smk_shs_ma', 'smk_shs__1', 'smk_shs__2', 'smk_shs__3', 'smk_shs_fe', 'smk_shs__4', 'smk_shs__5', 'smk_shs__6', 'geometry']


,DA_grp,comenvnm,AREA_KM2,NUM_DAs,DA16popest,DA16pop_m,DA16pop_f,CHBProp4,CHBPropF,id,...,smk_fmr__6,smk_shs_ma,smk_shs__1,smk_shs__2,smk_shs__3,smk_shs_fe,smk_shs__4,smk_shs__5,smk_shs__6,geometry
0,101,cape sable island,38.951757,5,2905.000000,1470.000000,1445.000000,12,12,101,...,0,0.197,0.129,0.286,1,0.146,0.100,0.203,0,"MULTIPOLYGON (((8319071.197 1242806.143, 83190..."
1,102,barrington,192.333328,6,3190.000000,1560.000000,1625.000000,12,12,102,...,0,0.171,0.113,0.243,0,0.162,0.120,0.219,1,"MULTIPOLYGON (((8304928.249 1246484.409, 83049..."
2,103,clyde river/welshtown,1112.782324,6,3157.972973,1607.297297,1545.675676,12,12,103,...,0,0.162,0.112,0.226,0,0.157,0.118,0.214,1,"MULTIPOLYGON (((8316721.603 1253855.563, 83167..."
3,104,shelburne,143.231560,7,2764.797297,1330.563063,1434.234234,12,12,104,...,1,0.170,0.106,0.253,0,0.146,0.103,0.203,0,"MULTIPOLYGON (((8335083.149 1286058.629, 83350..."
4,105,lockeport,1133.528856,5,1962.229730,957.139640,1005.090090,12,12,105,...,1,0.161,0.112,0.214,0,0.144,0.110,0.188,0,"MULTIPOLYGON (((8339571.991 1287376.237, 83395..."


In [ ]:
import geopandas as gpd
import pandas as pd
from shapely.geometry import Point
from io import StringIO
import requests

# Cargar datos NSCAF
@st.cache_data if 'streamlit' in dir() else lambda f: f
def cargar_nscaf():
    url = "https://data.novascotia.ca/api/views/tntn-er5g/rows.csv?accessType=DOWNLOAD"
    response = requests.get(url)
    return pd.read_csv(StringIO(response.text), low_memory=False)

# shapefile CE
gdf_ce = gpd.read_file(r"C:\Users\maria\OneDrive\Escritorio\HealthLocate\shapefiles\ComEnviron.shp")

# Health Atlas data
df_atlas = pd.read_csv(r"C:\Users\maria\OneDrive\Escritorio\HealthLocate\data\NSHealthAtlasDataEnvirons.csv")
df_atlas_clean = df_atlas[df_atlas['region'] == 'community-environs'].copy()

print("CE shapefile filas:", len(gdf_ce))
print("Atlas data filas:", len(df_atlas_clean))
print()

def get_patient_profile(numero, calle):
    # Search for an address in NSCAF
    df_nscaf = cargar_nscaf()
    resultado = df_nscaf[
        (df_nscaf['CIVICNUM'] == int(numero)) & 
        (df_nscaf['STRNAME'].str.upper() == calle.upper())
    ]
    
    if resultado.empty:
        print("Dirección no encontrada")
        return None
    
    lat = resultado.iloc[0]['LAT']
    lng = resultado.iloc[0]['LONG']
    comm = resultado.iloc[0]['COMM']
    print(f" Dirección: {numero} {calle}, {comm}")
    print(f"   Coordenadas: {lat}, {lng}")
    
    # Community Environ
    punto = gpd.GeoDataFrame([{'geometry': Point(lng, lat)}], crs="EPSG:4326")
    punto = punto.to_crs(gdf_ce.crs)
    
    ce_resultado = gpd.sjoin(punto, gdf_ce, how="left", predicate="within")
    
    if ce_resultado.empty:
        print("No se encontró Community Environ")
        return None
    
    ce_id = ce_resultado.iloc[0]['id']
    ce_name = ce_resultado.iloc[0]['name']
    print(f" Community Environ: {ce_name} (ID: {ce_id})")
    
    #  Connect to data from the Atlas
    atlas_row = df_atlas_clean[df_atlas_clean['id'] == ce_id]
    
    if atlas_row.empty:
        print("No se encontraron datos del Atlas para este CE")
        return None
    
    # Identify key indicators
    perfil = {
        'direccion': f"{numero} {calle}",
        'comunidad': comm,
        'community_environ': ce_name,
        'ce_id': ce_id,
        'msi_score': atlas_row.iloc[0]['msi-score2021'],
        'scs_score': atlas_row.iloc[0]['scs-score2021'],
        'sds_score': atlas_row.iloc[0]['sds-score2021'],
        'greenness': atlas_row.iloc[0]['green-pwndvi'],
        'air_quality_pm25': atlas_row.iloc[0]['aq-meanpm25'],
    }
    
    print(f"\n📊 Patient Profile:")
    for key, value in perfil.items():
        print(f"   {key}: {value}")
    
    return perfil

get_patient_profile("6281", "Jennings")

CE shapefile filas: 301
Atlas data filas: 300

 Dirección: 6281 Jennings, Halifax
   Coordenadas: 44.6404763084, -63.5956113907
 Community Environ: Dalhousie (ID: 9053)

📊 Patient Profile:
   direccion: 6281 Jennings
   comunidad: Halifax
   community_environ: Dalhousie
   ce_id: 9053
   msi_score: 4.0
   scs_score: 5.0
   sds_score: 5.0
   greenness: 0.52
   air_quality_pm25: 4.38


{'direccion': '6281 Jennings',
 'comunidad': 'Halifax',
 'community_environ': 'Dalhousie',
 'ce_id': np.int64(9053),
 'msi_score': np.float64(4.0),
 'scs_score': np.float64(5.0),
 'sds_score': np.float64(5.0),
 'greenness': np.float64(0.52),
 'air_quality_pm25': np.float64(4.38)}

: 